In [1]:
!pip install transformers torch datasets scikit-learn --quiet

import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, accuracy_score

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from tqdm.auto import tqdm

class CustomTextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

print("✅ Cell 1: Thư viện và lớp Dataset đã sẵn sàng.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 72.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 62.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 30.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 68.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 15.6 MB/s eta 0:00:00
ERROR: pip's dependency 

In [2]:
file_path = '/kaggle/input/vihallu-train/vihallu-train - vihallu-train.csv'
df = pd.read_csv(file_path)
df['text_input'] = df['prompt'] + " </s> " + df['response'] + " </s> " + df['context']
df = df[['text_input', 'label']].dropna().reset_index(drop=True)

label_encoder = LabelEncoder()
df['labels'] = label_encoder.fit_transform(df['label'])
num_labels = len(label_encoder.classes_)
id2label = {i: label for i, label in enumerate(label_encoder.classes_)}
label2id = {label: i for i, label in id2label.items()}

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["text_input"].tolist(),
    df["labels"].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df["labels"]
)

model_name = "vinai/phobert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)

print("Đang tokenize toàn bộ dữ liệu... Bước này sẽ mất vài phút.")
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=256)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=256)
print("✅ Tokenize hoàn tất!")

train_dataset = CustomTextDataset(train_encodings, train_labels)
val_dataset = CustomTextDataset(val_encodings, val_labels)

print("✅ Cell 2: Chuẩn bị dữ liệu hoàn tất.")

config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Đang tokenize toàn bộ dữ liệu... Bước này sẽ mất vài phút.
✅ Tokenize hoàn tất!
✅ Cell 2: Chuẩn bị dữ liệu hoàn tất.


In [3]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)
print("✅ DataLoader đã được tạo.")

model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=num_labels, id2label=id2label, label2id=label2id
)
print("✅ Model đã được tải.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Model đã được chuyển sang thiết bị: {device}")

optimizer = AdamW(model.parameters(), lr=2e-5)
print("✅ Optimizer đã được thiết lập.")

num_epochs = 5
print(f"Sẵn sàng huấn luyện trong {num_epochs} epochs.")

✅ DataLoader đã được tạo.


2025-10-08 11:23:08.040853: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759922588.204565      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759922588.250507      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/phobert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Model đã được tải.


model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Model đã được chuyển sang thiết bị: cuda
✅ Optimizer đã được thiết lập.
Sẵn sàng huấn luyện trong 5 epochs.


In [4]:
import os
import shutil

print("\n--- BẮT ĐẦU QUÁ TRÌNH HUẤN LUYỆN ---")

output_dir = "./all_checkpoints_in_one_folder" 
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

for epoch in range(num_epochs):
    model.train()
    total_train_loss = 0
    train_progress_bar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs} [Training]")
    
    for batch in train_progress_bar:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_train_loss += loss.item()
        loss.backward()
        optimizer.step()
        train_progress_bar.set_postfix({'loss': loss.item()})
        
    avg_train_loss = total_train_loss / len(train_loader)
    print(f"Epoch {epoch + 1} - Average Training Loss: {avg_train_loss:.4f}")
    
    temp_checkpoint_dir = os.path.join(output_dir, "temp")
    model.save_pretrained(temp_checkpoint_dir)
    tokenizer.save_pretrained(temp_checkpoint_dir)
    
    for filename in os.listdir(temp_checkpoint_dir):
        name, ext = os.path.splitext(filename)
        new_filename = f"{name}_epoch_{epoch + 1}{ext}"
        
        old_file_path = os.path.join(temp_checkpoint_dir, filename)
        new_file_path = os.path.join(output_dir, new_filename)
        
        shutil.move(old_file_path, new_file_path)
        
    shutil.rmtree(temp_checkpoint_dir)
    
    print(f"✅ Đã lưu các file checkpoint cho Epoch {epoch + 1} vào thư mục chính.")
    print("-" * 70)

print("\n--- HUẤN LUYỆN HOÀN TẤT ---")
print(f"Tất cả các file checkpoint đã được lưu tại: {output_dir}")

print("\nCác file trong thư mục checkpoint cuối cùng:")
!ls -l {output_dir}


--- BẮT ĐẦU QUÁ TRÌNH HUẤN LUYỆN ---


Epoch 1/5 [Training]:   0%|          | 0/175 [00:00<?, ?it/s]

Epoch 1 - Average Training Loss: 1.0827
✅ Đã lưu các file checkpoint cho Epoch 1 vào thư mục chính.
----------------------------------------------------------------------


Epoch 2/5 [Training]:   0%|          | 0/175 [00:00<?, ?it/s]

Epoch 2 - Average Training Loss: 0.7760
✅ Đã lưu các file checkpoint cho Epoch 2 vào thư mục chính.
----------------------------------------------------------------------


...

Epoch 3 - Average Training Loss: 0.6067
✅ Đã lưu các file checkpoint cho Epoch 3 vào thư mục chính.
----------------------------------------------------------------------


Epoch 4/5 [Training]:   0%|          | 0/175 [00:00<?, ?it/s]

Epoch 4 - Average Training Loss: 0.4713
✅ Đã lưu các file checkpoint cho Epoch 4 vào thư mục chính.
----------------------------------------------------------------------


Epoch 5/5 [Training]:   0%|          | 0/175 [00:00<?, ?it/s]

Epoch 5 - Average Training Loss: 0.3558
✅ Đã lưu các file checkpoint cho Epoch 5 vào thư mục chính.
----------------------------------------------------------------------

--- HUẤN LUYỆN HOÀN TẤT ---
Tất cả các file checkpoint đã được lưu tại: ./all_checkpoints_in_one_folder

Các file trong thư mục checkpoint cuối cùng:
total 2646896
-rw-r--r-- 1 root root        22 Oct  8 11:27 added_tokens_epoch_1.json
-rw-r--r-- 1 root root        22 Oct  8 11:32 added_tokens_epoch_2.json
-rw-r--r-- 1 root root        22 Oct  8 11:36 added_tokens_epoch_3.json
-rw-r--r-- 1 root root        22 Oct  8 11:41 added_tokens_epoch_4.json
-rw-r--r-- 1 root root        22 Oct  8 11:46 added_tokens_epoch_5.json
-rw-r--r-- 1 root root   1135173 Oct  8 11:27 bpe_epoch_1.codes
-rw-r--r-- 1 root root   1135173 Oct  8 11:32 bpe_epoch_2.codes
-rw-r--r-- 1 root root   1135173 Oct  8 11:36 bpe_epoch_3.codes
-rw-r--r-- 1 root root   1135173 Oct  8 11:41 bpe_epoch_4.codes
-rw-r--r-- 1 root root   1135173 Oct  8 11:46 bp

In [5]:
from sklearn.metrics import classification_report

print("\n--- BẮT ĐẦU ĐÁNH GIÁ TRÊN TẬP VALIDATION ---")

model.eval() 
all_preds = []
all_labels = []

with torch.no_grad(): 
    for batch in tqdm(val_loader, desc="Evaluating on Validation Set"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
        
class_names = label_encoder.classes_

print("\n--- Báo cáo Phân loại trên tập Validation ---")
report = classification_report(
    all_labels, 
    all_preds, 
    target_names=class_names,
    digits=4
)
print(report)


--- BẮT ĐẦU ĐÁNH GIÁ TRÊN TẬP VALIDATION ---


Evaluating on Validation Set:   0%|          | 0/44 [00:00<?, ?it/s]


--- Báo cáo Phân loại trên tập Validation ---
              precision    recall  f1-score   support

   extrinsic     0.7818    0.6529    0.7116       461
   intrinsic     0.6246    0.8082    0.7046       490
          no     0.8084    0.6860    0.7422       449

    accuracy                         0.7179      1400
   macro avg     0.7383    0.7157    0.7195      1400
weighted avg     0.7353    0.7179    0.7190      1400



In [6]:
from sklearn.metrics import classification_report
import pandas as pd

print("\n--- BẮT ĐẦU ĐÁNH GIÁ TRÊN TẬP TEST ---")

test_file_path = '/kaggle/input/testdata/data_test.csv' 

try:
    df_test = pd.read_csv(test_file_path)

    df_test['text_input'] = df_test['prompt'] + " </s> " + df_test['response'] + " </s> " + df_test['context']
    df_test = df_test[['text_input', 'label']].dropna().reset_index(drop=True)
    test_texts = df_test["text_input"].tolist()
    
    test_labels = label_encoder.transform(df_test["label"].tolist())

    print("Đang tokenize dữ liệu test...")
    test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=256)
    
    test_dataset = CustomTextDataset(test_encodings, test_labels)
    test_loader = DataLoader(test_dataset, batch_size=16)
    print("✅ Chuẩn bị dữ liệu test hoàn tất!")

    model.eval()
    all_preds_test = []
    all_labels_test = []

    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Evaluating on Test Set"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            all_preds_test.extend(preds)
            all_labels_test.extend(labels.cpu().numpy())

    print("\n--- Báo cáo Phân loại trên tập TEST ---")
    report_test = classification_report(
        all_labels_test, 
        all_preds_test, 
        target_names=class_names,
        digits=4
    )
    print(report_test)

except FileNotFoundError:
    print(f"LỖI: Không tìm thấy file test tại đường dẫn '{test_file_path}'.")
    print("Vui lòng kiểm tra lại và đảm bảo bạn đã thêm file test vào Kaggle dataset.")


--- BẮT ĐẦU ĐÁNH GIÁ TRÊN TẬP TEST ---
Đang tokenize dữ liệu test...
✅ Chuẩn bị dữ liệu test hoàn tất!


Evaluating on Test Set:   0%|          | 0/875 [00:00<?, ?it/s]


--- Báo cáo Phân loại trên tập TEST ---
              precision    recall  f1-score   support

   extrinsic     0.9281    0.8331    0.8780      4614
   intrinsic     0.8207    0.9257    0.8700      4896
          no     0.9350    0.9029    0.9186      4490

    accuracy                         0.8879     14000
   macro avg     0.8946    0.8872    0.8889     14000
weighted avg     0.8927    0.8879    0.8883     14000



In [7]:
save_directory = "./phobert-finetuned-model"

print(f"\nĐang lưu model và tokenizer vào thư mục: {save_directory}")

model.save_pretrained(save_directory)

tokenizer.save_pretrained(save_directory)

print(f"✅ Model và tokenizer đã được lưu thành công!")

print("\nCác file trong thư mục đã lưu:")
!ls -l {save_directory}


Đang lưu model và tokenizer vào thư mục: ./phobert-finetuned-model
✅ Model và tokenizer đã được lưu thành công!

Các file trong thư mục đã lưu:
total 529376
-rw-r--r-- 1 root root        22 Oct  8 11:53 added_tokens.json
-rw-r--r-- 1 root root   1135173 Oct  8 11:53 bpe.codes
-rw-r--r-- 1 root root       928 Oct  8 11:53 config.json
-rw-r--r-- 1 root root 540026460 Oct  8 11:53 model.safetensors
-rw-r--r-- 1 root root       167 Oct  8 11:53 special_tokens_map.json
-rw-r--r-- 1 root root      1203 Oct  8 11:53 tokenizer_config.json
-rw-r--r-- 1 root root    895321 Oct  8 11:53 vocab.txt
